# 13 — Verify the saved 0.943 output (CPU, no submission)

Attach the **exact saved Output version** of the notebook that scored Public
0.943, or a private Dataset containing its complete Output files/ZIP. Required
files include `btkd_v559_complete.json`, `_coat_raptor_blend_receipt.json`,
`submission.csv`, four CoAt child receipts, and `diagnostics/phase_events.jsonl`.

Set Accelerator **None**, Internet **Off**. Use Save & Run All. Inspect the
generated `rsna_0943_integrity/baseline_integrity.json`. Only
`VISIBLE_OUTPUT_PASS` lets us proceed; it validates the **visible saved run**,
not Kaggle's hidden scoring rerun. This notebook never changes or submits the
0.943 baseline and cannot calculate AUC from unlabeled test data.

In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
import math
import os
import zipfile

INPUT_ROOT = Path(os.environ.get('RSNA_AUDIT_INPUT', '/kaggle/input'))
SOURCE_OVERRIDE = os.environ.get('RSNA_AUDIT_SOURCE')  # Optional exact local path.
OUT = Path(os.environ.get('RSNA_AUDIT_OUTPUT', '/kaggle/working/rsna_0943_integrity'))
OUT.mkdir(parents=True, exist_ok=True)
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
EXPECTED_COATS = {'resgated_top3','global96_top3','d4_swa3','repairv1_top3'}

class OutputReader:
    def __init__(self, root):
        self.root = Path(root)
        self.archive = zipfile.ZipFile(self.root) if self.root.is_file() else None
        self.names = ([i.filename for i in self.archive.infolist() if not i.is_dir()]
                      if self.archive else
                      [str(p.relative_to(self.root)).replace('\\','/')
                       for p in self.root.rglob('*') if p.is_file()])
    def close(self):
        if self.archive: self.archive.close()
    def locate(self, suffix):
        hits = [n for n in self.names if n == suffix or n.endswith('/'+suffix)]
        if len(hits) > 1: raise RuntimeError(f'Ambiguous {suffix}: {hits}')
        return hits[0] if hits else None
    def read(self, suffix):
        name = self.locate(suffix)
        if name is None: return None
        return self.archive.read(name) if self.archive else (self.root/name).read_bytes()
    def json(self, suffix):
        raw = self.read(suffix)
        return json.loads(raw) if raw is not None else None

if SOURCE_OVERRIDE:
    sources = [Path(SOURCE_OVERRIDE)]
else:
    sources = []
    if not INPUT_ROOT.exists(): raise FileNotFoundError(INPUT_ROOT)
    for base, dirs, files in os.walk(INPUT_ROOT):
        depth = len(Path(base).relative_to(INPUT_ROOT).parts)
        dirs[:] = [d for d in dirs if d not in
                   {'train_series','test_series','study_cache','dicom','dicoms'}]
        if 'btkd_v559_complete.json' in files:
            sources.append(Path(base))
        for name in files:
            if name.lower().endswith('.zip'):
                candidate = Path(base)/name
                if candidate.stat().st_size <= 2*1024**3:
                    try:
                        with zipfile.ZipFile(candidate) as z:
                            if any(n == 'btkd_v559_complete.json' or
                                   n.endswith('/btkd_v559_complete.json')
                                   for n in z.namelist()):
                                sources.append(candidate)
                    except (OSError, zipfile.BadZipFile):
                        pass
        if depth >= 9: dirs[:] = []
sources = sorted(set(sources), key=str)
if len(sources) != 1:
    raise RuntimeError(f'Attach exactly one saved 0.943 Output, found: {sources}')

reader = OutputReader(sources[0])
checks = []
def check(name, ok, detail=None):
    checks.append({'check':name,'pass':bool(ok),'detail':detail})

try:
    receipt = reader.json('btkd_v559_complete.json')
    blend = reader.json('_coat_raptor_blend_receipt.json')
    raw_csv = reader.read('submission.csv')
    check('required_root_files', receipt is not None and blend is not None and raw_csv is not None)
    if receipt is None or blend is None or raw_csv is None:
        raise RuntimeError('Output is missing a required root file')
    table = list(csv.DictReader(io.StringIO(raw_csv.decode('utf-8-sig'))))
    columns = list(table[0]) if table else []
    ids = [row['StudyInstanceUID'] for row in table] if columns == ['StudyInstanceUID']+LABELS else []
    check('schema_and_unique_ids', bool(ids) and len(set(ids)) == len(ids),
          {'rows':len(table),'columns':columns})
    values = []
    try:
        values = [float(row[label]) for row in table for label in LABELS]
    except (KeyError, TypeError, ValueError):
        pass
    check('finite_probability_range', len(values) == len(table)*12 and
          all(math.isfinite(v) and 0 <= v <= 1 for v in values))
    check('submission_sha256', hashlib.sha256(raw_csv).hexdigest() ==
          receipt.get('submission_sha256'))
    check('complete_and_study_count', receipt.get('status') == 'COMPLETE' and
          receipt.get('cohort') == len(table) and blend.get('study_count') == len(table),
          {'cohort':receipt.get('cohort'),'rows':len(table)})
    check('dino_a5_raptor_counts', receipt.get('dino_members') == 20 and
          receipt.get('a5_folds') == 5 and receipt.get('raptor_views') == 4,
          {'dino':receipt.get('dino_members'),'a5':receipt.get('a5_folds'),
           'raptor':receipt.get('raptor_views')})
    members = blend.get('members', [])
    check('four_exact_coat_members', len(members) == 4 and set(members) == EXPECTED_COATS
          and receipt.get('coat_family_members') == 4, members)
    check('coat_probability_mean_then_rank',
          blend.get('family_reduction') == 'rank_of_member_probability_mean' and
          receipt.get('family_reduction') == 'rank_of_member_probability_mean')
    within = blend.get('within_coat', {})
    check('coat_four_equal_weights', set(within) == EXPECTED_COATS and
          all(abs(float(within[k])-0.25) < 1e-8 for k in EXPECTED_COATS))
    suspicious = [e for e in receipt.get('events', []) if any(
        marker in str(e.get('kind','')).lower() for marker in
        ('failed','failure','fallback','unavailable','rejected','neutral','partial'))]
    check('no_failure_or_fallback_events', not suspicious, suspicious[:10])
    child_names = ['_coat_arm_receipt.json','_coat_g96_receipt.json',
                   '_coat_repairv1_receipt.json','d4_input/d4.receipt.json']
    for child_name in child_names:
        child = reader.json(child_name)
        check('child_'+Path(child_name).name,
              child is not None and child.get('fallback_studies') == 0 and
              child.get('studies') == len(table) and
              str(child.get('status','')).startswith('VALID_'),
              None if child is None else {'status':child.get('status'),
                                           'studies':child.get('studies'),
                                           'fallback_studies':child.get('fallback_studies')})
    raw_phases = reader.read('diagnostics/phase_events.jsonl')
    phase_events = ([json.loads(line) for line in raw_phases.decode().splitlines()
                     if line.strip()] if raw_phases else [])
    phase_errors = [e for e in phase_events if e.get('status') == 'FAILED'
                    or e.get('ok') is False]
    check('phase_log_no_failure', bool(phase_events) and not phase_errors,
          phase_errors[:10])
    phases = receipt.get('phases', {})
    missing_or_wrong = []
    for name, info in phases.items():
        phase_path = 'diagnostics/'+Path(info.get('path','')).name
        raw = reader.read(phase_path)
        if raw is None or hashlib.sha256(raw).hexdigest() != info.get('sha256'):
            missing_or_wrong.append(name)
    check('all_phase_sha256', bool(phases) and not missing_or_wrong,
          {'phase_count':len(phases),'missing_or_wrong':missing_or_wrong})
    report = {'status':'VISIBLE_OUTPUT_PASS' if all(c['pass'] for c in checks) else 'FAIL',
              'source':str(sources[0]),'visible_studies':len(table),
              'hidden_scoring_run_verified':False,
              'source_elapsed_seconds':receipt.get('elapsed_seconds'),
              'checks':checks,
              'warning':'Visible Output is not the hidden Kaggle scoring rerun; no AUC or Public/Private prediction is inferred.'}
finally:
    reader.close()

path = OUT/'baseline_integrity.json'
path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('AUDIT STATUS:',report['status'],'visible studies:',report['visible_studies'])
print('Failed checks:',[x for x in checks if not x['pass']])
print('Saved:',path)